# Windows 11 Python Environment Setup for Data Fusion
This notebook automates the creation and setup of the `data_fusion_env_1` Python environment on your Windows 11 machine.

### Specs of Target System:
* **OS**: Windows 11
* **CPU**: AMD Ryzen 9
* **GPU**: NVIDIA GeForce RTX 3080
* **Target Environment Path**: `C:\python_envs\data_fusion_env_1`
* **Python Version**: 3.11

---
### Key Windows vs. macOS Differences Handled Here:
1. **PyTorch with CUDA**: Installs CUDA-enabled PyTorch (`cu121`) to leverage your NVIDIA RTX 3080 GPU.
2. **TensorFlow**: Installs the native Windows version of TensorFlow. Note that starting with TensorFlow 2.11, GPU support on native Windows is deprecated (TensorFlow runs on CPU). If you require GPU acceleration for TensorFlow, WSL2 (Windows Subsystem for Linux) is required.
3. **No `tensorflow-metal`**: Omitted since it is specific to Apple Silicon.
4. **Environment Location**: Configured to install in `C:\python_envs` rather than the macOS Homebrew directories.

---
### How to Run This Notebook:
1. Open this notebook in **VS Code** or **Cursor**.
2. Select your current default/base python kernel or Conda interpreter in the top-right corner to execute these setup cells.
3. Run the cells sequentially.

In [ ]:
import os
import sys
import subprocess

def run_command(command, shell=True):
    """Runs a system command and streams output to the notebook cell in real-time."""
    print(f"--- Running: {command} ---\n")
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=shell, text=True)
    
    # Stream output line by line
    for line in process.stdout:
        print(line, end="")
    
    process.wait()
    if process.returncode != 0:
        print(f"\n[ERROR] Command failed with exit code: {process.returncode}")
        return False
    else:
        print(f"\n[SUCCESS] Completed successfully.")
        return True

# Ensure target directory base folder exists
try:
    os.makedirs(r"C:\python_envs", exist_ok=True)
    print("[SUCCESS] C:\\python_envs directory is ready.")
except Exception as e:
    print(f"[WARNING] Could not create C:\\python_envs directory: {e}")
    print("If you encounter permission issues, please run VS Code/Cursor as Administrator, or choose a folder you have write access to.")

## Step 1: Verify Conda Installation
Let's check if `conda` is accessible in the current system path.

In [ ]:
# Check if conda is available in the current environment path
import shutil

conda_path = shutil.which("conda")
if conda_path:
    print(f"Conda found at: {conda_path}")
    run_command("conda --version")
else:
    print("[ERROR] 'conda' command was not found in your PATH.")
    print("Please make sure Anaconda or Miniconda is installed, and that it is added to your PATH or run from a Conda Prompt.")

## Step 2: Configure Conda and Create the Environment
We will prepend `C:\python_envs` to Conda's environment directories (`envs_dirs`) so that conda knows where to look for environment names. Then we will create the environment `data_fusion_env_1` with Python 3.11.

In [ ]:
# Prepend target folder to envs_dirs
run_command("conda config --prepend envs_dirs C:\\python_envs")

# Create Python 3.11 environment in C:\python_envs\data_fusion_env_1
env_path = r"C:\python_envs\data_fusion_env_1"
run_command(f"conda create -p {env_path} python=3.11 -y")

## Step 3: Install Conda-Forge Packages
We will install the required conda-forge packages into the environment using its path.

In [ ]:
conda_packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "matplotlib",
    "pyserial",
    "tqdm",
    "joblib",
    "ipykernel",
    "jupyterlab",
    "notebook",
    "pyyaml",
    "h5py",
    "pyarrow",
    "filterpy",
    "tensorboard"
]

# Install all packages using the specific environment prefix
packages_str = " ".join(conda_packages)
run_command(f"conda install -p C:\\python_envs\\data_fusion_env_1 -c conda-forge {packages_str} -y")

## Step 4: Install CUDA-Enabled PyTorch via Pip
We will install PyTorch with GPU support using the custom wheel index for CUDA 12.1, which is optimized for your RTX 3080 GPU.

In [ ]:
# Specify path to the new environment's pip executable to avoid activation issues
env_pip = r"C:\python_envs\data_fusion_env_1\Scripts\pip"

# Run pip install with CUDA index
run_command(f'"{env_pip}" install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')

## Step 5: Install OpenCV, Mediapipe, and Input Simulation Packages
We'll install `opencv-python`, `mediapipe`, `pyautogui`, and `pynput` via pip.

In [ ]:
pip_packages = [
    "opencv-python",
    "mediapipe",
    "pyautogui",
    "pynput"
]

packages_str = " ".join(pip_packages)
run_command(f'"{env_pip}" install {packages_str}')

## Step 6: Install TensorFlow (CPU-only on Windows native)
As noted, native Windows TensorFlow >= 2.11 does not support GPU acceleration directly. We will install standard `tensorflow` which runs on CPU. If GPU TensorFlow is required later, we recommend setting up the project within WSL2.

In [ ]:
# Install tensorflow (note: tensorflow-metal is omitted on Windows)
run_command(f'"{env_pip}" install tensorflow')

## Step 7: Register Jupyter Kernel
To make this new environment selectable in Jupyter notebooks (including this one, VS Code, and JupyterLab), we register it as a user kernel.

In [ ]:
env_python = r"C:\python_envs\data_fusion_env_1\python"
run_command(f'"{env_python}" -m ipykernel install --user --name data_fusion_env_1 --display-name "Python (data_fusion_env_1)"') 

## Step 8: Verification
Let's run a verification script using the environment's python interpreter to confirm that all key libraries are successfully installed and checking if the RTX 3080 GPU is correctly detected by PyTorch!

In [ ]:
verification_code = """
import sys
print('=== ENV VERIFICATION ===')
print('Python Path:', sys.executable)
print('Python Version:', sys.version)

try:
    import numpy as np
    print('NumPy:', np.__version__)
except Exception as e:
    print('NumPy failed:', e)

try:
    import torch
    print('PyTorch Version:', torch.__version__)
    print('CUDA available in PyTorch:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA Device Count:', torch.cuda.device_count())
        print('CUDA Device Name (RTX 3080):', torch.cuda.get_device_name(0))
except Exception as e:
    print('PyTorch failed:', e)

try:
    import tensorflow as tf
    print('TensorFlow Version:', tf.__version__)
    print('TensorFlow GPU List:', tf.config.list_physical_devices("GPU"))
except Exception as e:
    print('TensorFlow failed:', e)

try:
    import cv2
    print('OpenCV:', cv2.__version__)
except Exception as e:
    print('OpenCV failed:', e)

try:
    import mediapipe as mp
    print('MediaPipe:', mp.__version__)
except Exception as e:
    print('MediaPipe failed:', e)
"""

# Run verification using the target environment's Python
import tempfile
with tempfile.NamedTemporaryFile(suffix=".py", delete=False) as temp_file:
    temp_file.write(verification_code.encode("utf-8"))
    temp_file_name = temp_file.name

try:
    run_command(f'"{env_python}" "{temp_file_name}"')
finally:
    if os.path.exists(temp_file_name):
        os.remove(temp_file_name)
